In [1]:
import os
import warnings
import pandas as pd
import numpy as np
import scanpy as sc
import seaborn as sns
import matplotlib as mpl
import matplotlib.pyplot as plt
from pathlib import Path
from scipy.stats import mannwhitneyu
from statsmodels.stats.multitest import multipletests
warnings.filterwarnings("ignore")

# --- 2. DEFINE DIRECTORIES & NEW REPO PATHS ---
WORKING_DIR = "/data/ag_reichart_lindberg/nhadizad/nastaran/projects/xeno/analysis/v1/GSS/OSM_FB/"
os.makedirs(WORKING_DIR, exist_ok=True)
os.chdir(WORKING_DIR)

TARGET_CONDITIONS = [
    "Control",
    "Prematurely-terminated",
    "3month-protocol",
    "6month-protocol",
    "PCMV/PRV",
    "Rejection",
]

COND_PALETTE = {
    "Control":                "#EFE48E",
    "3month-protocol":        "skyblue",
    "6month-protocol":        "#08528f",
    "Prematurely-terminated": "#C9B8D9",
    "PCMV/PRV":               "#F0C096",
    "Rejection":              "#E09E9E",
}


In [2]:
# Plotting configs

mpl.rcParams["pdf.fonttype"]       = 42
mpl.rcParams["ps.fonttype"]        = 42
sc.settings.set_figure_params(dpi=300, dpi_save=300)
mpl.rcParams["font.family"]        = "sans-serif"
mpl.rcParams["font.sans-serif"]    = ["Arial"]
mpl.rcParams["font.size"]          = 5
mpl.rcParams["font.weight"]        = "bold"
mpl.rcParams["axes.labelweight"]   = "bold"
mpl.rcParams["axes.titleweight"]   = "bold"
mpl.rcParams["figure.titleweight"] = "bold"
mpl.rcParams["axes.grid"]          = False
mpl.rcParams["axes.spines.top"]    = False
mpl.rcParams["axes.spines.right"]  = False
sns.set_style("ticks", {"axes.grid": False})

GLOBAL_OBJ_PATH = "/data/ag_reichart_lindberg/nhadizad/nastaran/projects/xeno/analysis/v1/cellchat/v1/merging_UMIs/Raw_UMI_merged_global_object_preclinical.h5ad"
RESULTS_DIR     = "/data/ag_reichart_lindberg/nhadizad/nastaran/projects/xeno/analysis/v1/GSS/OSM_FB"
os.makedirs(RESULTS_DIR, exist_ok=True)

BOXPLOT_FIGSIZE      = (4.25, 3.25)
BOXPLOT_WIDTH        = 0.9
BOXPLOT_LINEWIDTH    = 0.9
MEDIAN_LINEWIDTH     = 0.7
STRIP_DOT_SIZE       = 3
STRIP_ALPHA          = 0.5
BRACKET_LINE_WIDTH   = 0.9
BRACKET_OFFSET_FRAC  = 0.05
BRACKET_TICK_FRAC    = 0.02
STAR_GAP_FRAC        = 0.01
STAR_FONT_SIZE       = 7
DPI                  = 300


TRANSPLANT_GROUP_PALETTE = {
    "Non-transplanted": "#4c72b0", # Muted blue
    "Xeno":             "#dd8452"  # Muted orange
}

# Metadata cols
CONDITION_COL  = "Condition"
SAMPLE_COL     = "Sample_ID"
CELL_TYPE_COL  = "cell_type"
CELL_STATE_COL = "cell_state"


In [3]:

# Let's define the OSM gene list from the publication
OSM_CORE_GENES = [
    'OSM', 'OSMR', 'IL6ST', 'LIFR',
    'JAK1', 'JAK2', 'TYK2',
    'STAT1', 'STAT3', 'STAT5A',
    'STAT5B', 'STAT6',
    'SOCS3', 'PIAS3'
]

EXPR_THRESHOLD = 0.0125

print("Loading global object...")
adata = sc.read(GLOBAL_OBJ_PATH)
print(f"  Full object: {adata.shape[0]} cells x {adata.shape[1]} genes")

# 1. Normalize and log1p the ENTIRE object first
print("Normalizing and log1p transforming global object...")
adata_gss = adata.copy()
sc.pp.normalize_total(adata_gss, target_sum=1e4)
sc.pp.log1p(adata_gss)

# 2. Extract strictly the Fibroblast (FB) fraction
print("Filtering to FB cell type...")
fb_mask = adata_gss.obs[CELL_TYPE_COL] == 'FB'
adata_fb = adata_gss[fb_mask].copy()

# Ensure we only keep target conditions
adata_fb = adata_fb[adata_fb.obs[CONDITION_COL].isin(TARGET_CONDITIONS)].copy()

# Enforce condition ordering in the object
adata_fb.obs[CONDITION_COL] = pd.Categorical(
    adata_fb.obs[CONDITION_COL],
    categories=[c for c in TARGET_CONDITIONS if c in adata_fb.obs[CONDITION_COL].unique()],
    ordered=True
)

print(f"\n  Fibroblast cells: {adata_fb.n_obs}")
print(f"  Cell states present: {sorted(adata_fb.obs[CELL_STATE_COL].unique().tolist())}")

print("\n--- Filtering and Scoring OSM Core Genes ---")
genes_in_data = [g for g in OSM_CORE_GENES if g in adata_fb.var_names]
genes_missing = [g for g in OSM_CORE_GENES if g not in adata_fb.var_names]

if genes_missing:
    print(f"  Not in var_names ({len(genes_missing)}): {', '.join(genes_missing)}")

# Expression threshold filter
expr_df = sc.get.obs_df(adata_fb, keys=genes_in_data + [CELL_STATE_COL])
max_mean = expr_df.groupby(CELL_STATE_COL, observed=True).mean().max()
retained_genes = max_mean[max_mean >= EXPR_THRESHOLD].index.tolist()

print(f"  Scoring {len(retained_genes)} retained OSM core genes...")
sc.tl.score_genes(adata_fb, gene_list=retained_genes, score_name='osm_core')

scored_path = os.path.join(RESULTS_DIR, "OSM_FB_scored.h5ad")
adata_fb.write(scored_path)
print(f"Saved scored FB object: {scored_path}")

# Prepare master scoring dataframe
scores_df = adata_fb.obs[['osm_core', SAMPLE_COL, CELL_STATE_COL, CONDITION_COL]].copy()


Loading global object...
  Full object: 233200 cells x 40653 genes
Normalizing and log1p transforming global object...
Filtering to FB cell type...

  Fibroblast cells: 15562
  Cell states present: ['Angio_FB', 'ECM_FB', 'Homeo_FB', 'Hypoxia_FB', 'ISG_FB', 'Myo_FB', 'Progenitor_FB', 'Prolif_Myo_FB']

--- Filtering and Scoring OSM Core Genes ---
  Scoring 13 retained OSM core genes...
Saved scored FB object: /data/ag_reichart_lindberg/nhadizad/nastaran/projects/xeno/analysis/v1/GSS/OSM_FB/OSM_FB_scored.h5ad


In [4]:
adata_fb.obs.Exp

5411-2_CTATCAAGTCAACCAT-1    Control
5411-2_CTAACCTTCGATCATG-1    Control
5411-2_ATCAACTAGCGAGACA-1    Control
5411-2_CACGAAAGTCCATACG-1    Control
5411-2_ACGGGCTCACCTTGAC-1    Control
                              ...   
PAV89_GGCATGGGTCAAGGAC-1      p_xHTx
PAV89_TGAGTTACAAATGGCT-1      p_xHTx
PAV89_AACTCTTCACCTATGC-1      p_xHTx
PAV89_ATGGACCTCGCTAGGT-1      p_xHTx
PAV89_ACCATCAAGTTACGGC-1      p_xHTx
Name: Exp, Length: 15562, dtype: category
Categories (2, object): ['Control', 'p_xHTx']

# OSM scores per states

In [5]:
print("\nGenerating Plot 1: Per Cell State Overview...")
plot1_dir = os.path.join(RESULTS_DIR, "plot1_per_cell_state")
os.makedirs(plot1_dir, exist_ok=True)

# Pseudobulk per Sample & Cell State
agg_state = scores_df.groupby([SAMPLE_COL, CELL_STATE_COL], observed=True)['osm_core'].median().reset_index()
agg_state.to_csv(os.path.join(plot1_dir, "osm_core_per_cellstate_data.csv"), index=False)

fig, ax = plt.subplots(figsize=(6, 4))
sns.boxplot(
    data=agg_state, x=CELL_STATE_COL, y='osm_core', color='white',
    boxprops=dict(edgecolor='black', linewidth=BOXPLOT_LINEWIDTH),
    medianprops=dict(color='black', linewidth=MEDIAN_LINEWIDTH),
    whiskerprops=dict(color='black', linewidth=BOXPLOT_LINEWIDTH),
    capprops=dict(color='black', linewidth=BOXPLOT_LINEWIDTH),
    showfliers=False, ax=ax
)
sns.stripplot(
    data=agg_state, x=CELL_STATE_COL, y='osm_core', color='black',
    size=STRIP_DOT_SIZE, alpha=STRIP_ALPHA, ax=ax
)

ax.set_title('OSM Core Score per FB Cell State', fontsize=10, fontweight='bold')
ax.set_xlabel('Cell State', fontsize=8, fontweight='bold')
ax.set_ylabel('OSM Core Score (Sample Median)', fontsize=8, fontweight='bold')
plt.xticks(rotation=45, ha='right', fontsize=7, fontweight='bold')
plt.yticks(fontsize=7, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(plot1_dir, "OSM_Score_Per_CellState.pdf"), dpi=DPI, bbox_inches="tight")
plt.close()



Generating Plot 1: Per Cell State Overview...


# Boxplot & KDE + cell state X condition boxplots with stats

In [6]:

print("Generating Plot 2 & 4: Per Condition Overview & KDE...")
plot2_dir = os.path.join(RESULTS_DIR, "plot2_per_condition")
os.makedirs(plot2_dir, exist_ok=True)

# Pseudobulk per Sample & Condition
agg_cond = scores_df.groupby([SAMPLE_COL, CONDITION_COL], observed=True)['osm_core'].median().reset_index()
agg_cond.to_csv(os.path.join(plot2_dir, "osm_core_per_condition_data.csv"), index=False)

# 5A. Boxplot
fig, ax = plt.subplots(figsize=(3, 2.75))
sns.boxplot(
    data=agg_cond, x=CONDITION_COL, y='osm_core', order=TARGET_CONDITIONS, palette=COND_PALETTE,
    boxprops=dict(edgecolor='black', linewidth=BOXPLOT_LINEWIDTH),
    medianprops=dict(color='black', linewidth=MEDIAN_LINEWIDTH),
    whiskerprops=dict(color='black', linewidth=BOXPLOT_LINEWIDTH),
    capprops=dict(color='black', linewidth=BOXPLOT_LINEWIDTH),
    showfliers=False, ax=ax
)
sns.stripplot(
    data=agg_cond, x=CONDITION_COL, y='osm_core', order=TARGET_CONDITIONS, color='black',
    size=STRIP_DOT_SIZE, alpha=STRIP_ALPHA, ax=ax
)

ax.set_title('OSM Core Score per Condition', fontsize=10, fontweight='bold')
ax.set_xlabel('Condition', fontsize=8, fontweight='bold')
ax.set_ylabel('OSM Core Score (Sample Median)', fontsize=8, fontweight='bold')
plt.xticks(rotation=45, ha='right', fontsize=7, fontweight='bold')
plt.yticks(fontsize=7, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(plot2_dir, "OSM_Score_Per_Condition_Boxplot.pdf"), dpi=DPI, bbox_inches="tight")
plt.close()

# 5B. KDE Ridge Plot (Ordered by Median)
order_by_median = agg_cond.groupby(CONDITION_COL, observed=True)['osm_core'].median().sort_values().index.tolist()
agg_cond_kde = agg_cond.copy()
agg_cond_kde[CONDITION_COL] = pd.Categorical(agg_cond_kde[CONDITION_COL], categories=order_by_median, ordered=True)

g = sns.FacetGrid(
    agg_cond_kde, row=CONDITION_COL, hue=CONDITION_COL, aspect=4, height=1.0, 
    sharex=True, sharey=False, palette=COND_PALETTE
)
g.map(sns.kdeplot, 'osm_core', fill=True, alpha=0.95, bw_adjust=0.8, cut=0)
g.map(sns.kdeplot, 'osm_core', color='black', lw=0.8, bw_adjust=0.8, cut=0)

for ax, cond in zip(g.axes.flat, order_by_median):
    m = agg_cond_kde.loc[agg_cond_kde[CONDITION_COL]==cond, 'osm_core'].median()
    ax.axvline(m, ls='--', lw=1.3, color='black', alpha=0.5)
    ax.set_ylabel('')
    ax.set_yticks([])

g.set_xlabels('OSM Core Score (Sample Median)', fontsize=8, fontweight='bold')
g.fig.subplots_adjust(top=0.85)
g.fig.suptitle('OSM Core Score Distribution by Condition (Ordered by Median)', fontsize=10, fontweight='bold')
sns.despine(left=True, bottom=False)
plt.savefig(os.path.join(plot2_dir, "OSM_Score_Per_Condition_KDE.pdf"), dpi=DPI, bbox_inches="tight")
plt.close()

# 6. PLOT 3: Cell state x condition + statistics

print("Generating Plot 3: Cell State x Condition with Stats...")
plot3_dir = os.path.join(RESULTS_DIR, "plot3_cellstate_x_condition")
os.makedirs(plot3_dir, exist_ok=True)

# Pseudobulk per Sample, Condition, & Cell State
agg_cs_cond = scores_df.groupby([SAMPLE_COL, CONDITION_COL, CELL_STATE_COL], observed=True)['osm_core'].median().reset_index()

active_states = sorted(agg_cs_cond[CELL_STATE_COL].unique())
all_cs_stats = []

for cs in active_states:
    cs_data = agg_cs_cond[agg_cs_cond[CELL_STATE_COL] == cs].copy()
    if cs_data.empty: continue
    safe_cs = cs.replace("/", "_").replace(" ", "_")
    
    # Statistical Testing (Control vs Others within this Cell State)
    cs_stats = []
    cs_pvals = []
    
    control_scores = cs_data[cs_data[CONDITION_COL] == 'Control']['osm_core'].dropna()
    med_control = control_scores.median() if len(control_scores) > 0 else np.nan
    
    for cond in [c for c in TARGET_CONDITIONS if c != 'Control']:
        test_scores = cs_data[cs_data[CONDITION_COL] == cond]['osm_core'].dropna()
        med_test = test_scores.median() if len(test_scores) > 0 else np.nan
        log2fc = np.log2(med_test / med_control) if med_control > 0 and med_test > 0 else np.nan
        
        pval = np.nan
        if len(control_scores) >= 2 and len(test_scores) >= 2:
            try:
                _, pval = mannwhitneyu(test_scores, control_scores, alternative="two-sided")
            except: pass
            
        cs_stats.append({
            "Cell_State": cs, "Reference": "Control", "Condition": cond,
            "N_Ref": len(control_scores), "N_Cond": len(test_scores),
            "Median_Ref": med_control, "Median_Cond": med_test,
            "log2FC_vs_Control": log2fc, "pvalue": pval
        })
        if not np.isnan(pval): cs_pvals.append(pval)
            
    cs_stats_df = pd.DataFrame(cs_stats)
    cs_stats_df['FDR'] = np.nan
    if cs_pvals:
        valid_mask = cs_stats_df["pvalue"].notna()
        _, fdr_vals, _, _ = multipletests(cs_stats_df.loc[valid_mask, "pvalue"], method="fdr_bh")
        cs_stats_df.loc[valid_mask, "FDR"] = fdr_vals
        
    cs_stats_df["Significance"] = cs_stats_df["FDR"].apply(
        lambda f: "***" if f < 0.001 else ("**" if f < 0.01 else ("*" if f < 0.05 else "")) if not np.isnan(f) else ""
    )
    cs_stats_df.to_csv(os.path.join(plot3_dir, f"{safe_cs}_stats.csv"), index=False)
    all_cs_stats.append(cs_stats_df)

    fig, ax = plt.subplots(figsize=(5, 3.5))
    sns.boxplot(
        data=cs_data, x=CONDITION_COL, y='osm_core', order=TARGET_CONDITIONS, palette=COND_PALETTE,
        width=BOXPLOT_WIDTH, showfliers=False, ax=ax,
        boxprops=dict(edgecolor='black', linewidth=BOXPLOT_LINEWIDTH),
        medianprops=dict(color='black', linewidth=MEDIAN_LINEWIDTH),
        whiskerprops=dict(color='black', linewidth=BOXPLOT_LINEWIDTH),
        capprops=dict(color='black', linewidth=BOXPLOT_LINEWIDTH),
    )
    sns.stripplot(
        data=cs_data, x=CONDITION_COL, y='osm_core', order=TARGET_CONDITIONS, 
        color='black', size=STRIP_DOT_SIZE, alpha=STRIP_ALPHA, ax=ax
    )
    
    # Add significance brackets vs Control
    for _, row in cs_stats_df.iterrows():
        if row["Significance"] == "": continue
        if row["Condition"] not in TARGET_CONDITIONS: continue
        
        idx_control = TARGET_CONDITIONS.index("Control")
        idx_test = TARGET_CONDITIONS.index(row["Condition"])
        
        y_max = cs_data['osm_core'].max()
        y_range = y_max - cs_data['osm_core'].min() if y_max - cs_data['osm_core'].min() > 0 else 1.0
        
        # Step brackets up to avoid overlap
        bracket_y = y_max + (BRACKET_OFFSET_FRAC * y_range) + (idx_test * 0.05 * y_range)
        tick_h = BRACKET_TICK_FRAC * y_range
        
        ax.plot([idx_control, idx_control, idx_test, idx_test], 
                [bracket_y, bracket_y + tick_h, bracket_y + tick_h, bracket_y], 
                lw=BRACKET_LINE_WIDTH, color="black")
        ax.text((idx_control + idx_test)/2, bracket_y + tick_h + STAR_GAP_FRAC * y_range, 
                row["Significance"], ha="center", va="bottom", fontsize=STAR_FONT_SIZE, fontweight="bold")

    ax.set_title(f"OSM Score — {cs}", fontweight="bold", fontsize=9)
    ax.set_xlabel("Condition", fontweight="bold", fontsize=8)
    ax.set_ylabel("OSM Score (Sample Median)", fontweight="bold", fontsize=8)
    plt.xticks(rotation=45, ha="right", fontweight="bold", fontsize=7)
    plt.yticks(fontweight="bold", fontsize=7)
    plt.tight_layout()
    plt.savefig(os.path.join(plot3_dir, f"OSM_Score_{safe_cs}_Condition_Boxplot.pdf"), dpi=DPI, bbox_inches="tight")
    plt.close()

if all_cs_stats:
    pd.concat(all_cs_stats, ignore_index=True).to_csv(os.path.join(plot3_dir, "ALL_CellStates_Condition_Stats.csv"), index=False)

Generating Plot 2 & 4: Per Condition Overview & KDE...
Generating Plot 3: Cell State x Condition with Stats...


# Super groupping: All xenografts vs non-transplanted control hearts

In [5]:
import os
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import mannwhitneyu

RESULTS_DIR = "/data/ag_reichart_lindberg/nhadizad/nastaran/projects/xeno/analysis/v1/GSS/OSM_FB"
plot5_dir = os.path.join(RESULTS_DIR, "plot5_exp_group")
os.makedirs(plot5_dir, exist_ok=True)

adata_fb = sc.read ('/data/ag_reichart_lindberg/nhadizad/nastaran/projects/xeno/analysis/v1/GSS/OSM_FB/OSM_FB_scored.h5ad')

SAMPLE_COL = "Sample_ID"
EXP_COL    = "Exp" 

EXP_PALETTE = {
    # "Control": "#EFE48E", # Control yellow 
    # "p_xHTx":  "#dd8452"  # Muted orange 

    "Control": "#EFE48E", # Control yellow 
    "p_xHTx":  "#B76E79"  # Muted pink 

}
EXP_ORDER = ["Control", "p_xHTx"]

# Visualization configs
FRAME_W_IN, FRAME_H_IN, BRACKET_BAND_PT = 3.30, 1.271, 21
DATA_H_IN = FRAME_H_IN - BRACKET_BAND_PT / 72
VIOLIN_WIDTH = 0.9          # violin width as a fraction of the group spacing (seaborn default 0.8)
MARGIN_IN  = dict(left=0.62, right=0.08, top=0.30, bottom=0.40)
STRIP_DOT_SIZE = 3
STRIP_ALPHA    = 0.6
DPI            = 300

print(f"Generating Plot 5: Exp Group Violin Plot using '{EXP_COL}' column...")

# Extract relevant data directly from the scored object
scores_exp_df = adata_fb.obs[['osm_core', SAMPLE_COL, EXP_COL]].copy()
scores_exp_df = scores_exp_df.dropna(subset=[SAMPLE_COL, EXP_COL])
scores_exp_df[EXP_COL] = pd.Categorical(scores_exp_df[EXP_COL], categories=EXP_ORDER, ordered=True)

# Pseudobulk: Aggregate per Sample (Median)
agg_exp = scores_exp_df.groupby([SAMPLE_COL, EXP_COL], observed=True)['osm_core'].median().reset_index()

# Save the aggregated data
agg_exp.to_csv(os.path.join(plot5_dir, "exp_group_data.csv"), index=False)

# Statistical tetsing: Mann whitney U test with BH FDR adjustment

control_scores = agg_exp[agg_exp[EXP_COL] == 'Control']['osm_core'].dropna()
pxhtx_scores   = agg_exp[agg_exp[EXP_COL] == 'p_xHTx']['osm_core'].dropna()

pval_exp = np.nan
if len(control_scores) >= 2 and len(pxhtx_scores) >= 2:
    _, pval_exp = mannwhitneyu(pxhtx_scores, control_scores, alternative="two-sided")

stats_exp_df = pd.DataFrame([{
    "Comparison": "p_xHTx vs Control",
    "N_Control": len(control_scores), "N_p_xHTx": len(pxhtx_scores),
    "Median_Control": control_scores.median() if len(control_scores) > 0 else np.nan,
    "Median_p_xHTx": pxhtx_scores.median() if len(pxhtx_scores) > 0 else np.nan,
    "pvalue": pval_exp
}])
stats_exp_df.to_csv(os.path.join(plot5_dir, "exp_group_stats.csv"), index=False)

# Violin plot params
_W = MARGIN_IN["left"] + FRAME_W_IN + MARGIN_IN["right"]; _H = MARGIN_IN["bottom"] + FRAME_H_IN + MARGIN_IN["top"]
fig = plt.figure(figsize=(_W, _H))
ax = fig.add_axes([MARGIN_IN["left"] / _W, MARGIN_IN["bottom"] / _H, FRAME_W_IN / _W, FRAME_H_IN / _H])

# Draw the violin plot
sns.violinplot(
    data=agg_exp, x=EXP_COL, y='osm_core', order=EXP_ORDER,
    palette=EXP_PALETTE, inner='quartile', linewidth=1.5, width=VIOLIN_WIDTH, ax=ax
)

# Overlay the jitter dots
sns.stripplot(
    data=agg_exp, x=EXP_COL, y='osm_core', order=EXP_ORDER,
    color='black', size=STRIP_DOT_SIZE, alpha=STRIP_ALPHA, ax=ax, jitter=True
)

ax.margins(y=0.02); ax.autoscale_view(scalex=False)        
_lo, _hi = ax.get_ylim()                                 
_unit = (_hi - _lo) / DATA_H_IN                           
ax.set_ylim(_lo, _hi + (BRACKET_BAND_PT / 72) * _unit)
if not np.isnan(pval_exp) and pval_exp < 0.05:
    sig_str = "***" if pval_exp < 0.001 else ("**" if pval_exp < 0.01 else "*")
    import matplotlib.patheffects as pe
    yb = _hi + (3 / 72) * _unit; h = (3 / 72) * _unit
    ax.plot([0, 0, 1, 1], [yb, yb + h, yb + h, yb], color="#333333", lw=0.8, clip_on=False, zorder=5)
    ax.annotate(sig_str, (0.5, yb + h), xytext=(0, 1.0), textcoords="offset points", ha="center", va="bottom", zorder=6,
                fontsize=11, fontweight="bold", color="#111111", path_effects=[pe.withStroke(linewidth=0.9, foreground="#111111")])

ax.set_title("OSM Core Score by Exp Group", fontsize=10, fontweight='bold')
ax.set_xlabel("Experimental Group", fontsize=8, fontweight='bold')
ax.set_ylabel("OSM Core Score\n(Sample Median)", fontsize=8, fontweight='bold')
plt.xticks(fontsize=8, fontweight='bold')
plt.yticks(fontsize=7, fontweight='bold')

# Save the plot
pdf_path = os.path.join(plot5_dir, "OSM_Score_Exp_Group_Violin.pdf")
plt.savefig(pdf_path, dpi=DPI, bbox_inches="tight")
_p = ax.get_position(); print(f"frame (boxed axes area): {_p.width * _W:.3f} in wide x {_p.height * _H:.3f} in tall")
plt.close()

print(f"\nSUCCESS: Plot 5 (Exp Group Violin) and stats saved to:\n{plot5_dir}")
print("\n" + "=" * 70)
print("OSM FIBROBLAST SCORING PIPELINE COMPLETE")
print(f"All results saved in: {RESULTS_DIR}")
print("=" * 70)


Generating Plot 5: Exp Group Violin Plot using 'Exp' column...


frame (boxed axes area): 3.300 in wide x 1.271 in tall

SUCCESS: Plot 5 (Exp Group Violin) and stats saved to:
/data/ag_reichart_lindberg/nhadizad/nastaran/projects/xeno/analysis/v1/GSS/OSM_FB/plot5_exp_group

OSM FIBROBLAST SCORING PIPELINE COMPLETE
All results saved in: /data/ag_reichart_lindberg/nhadizad/nastaran/projects/xeno/analysis/v1/GSS/OSM_FB
